# This Section Concerns all mentioned Datasets with Boost

# Imports

In [26]:
# Standard library
import os

# Data handling
import numpy as np
import pandas as pd
from scipy import sparse
import cupy as cp
from cuml.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier


def to_gpu_dense(X):
    """
    Converts input matrix X into a dense CuPy array safely.
    - If X is SciPy CSR/CSC/COO sparse → convert to NumPy dense → CuPy
    - If X is pandas DataFrame → convert to NumPy → CuPy
    - If X is NumPy array → CuPy
    - Never return nested object-arrays
    """

    # Case 1: SciPy sparse (CSR, CSC, COO)
    if sparse.issparse(X):
        # Convert sparse → dense NumPy → CuPy
        X_np = X.toarray().astype(np.float32)
        return cp.asarray(X_np)

    # Case 2: Pandas DataFrame
    if hasattr(X, "values"):
        return cp.asarray(X.values.astype(np.float32))

    # Case 3: NumPy array
    if isinstance(X, np.ndarray):
        return cp.asarray(X.astype(np.float32))

    # If it's already CuPy
    if isinstance(X, cp.ndarray):
        return X

    raise TypeError(f"Unsupported type passed to to_gpu_dense(): {type(X)}")

def load_split(data, split, base_path="../data/splits/"):
    """
    Loads X_train, X_test, y_train, y_test for a given dataset + split.
    Automatically detects whether features are stored as sparse (.npz)
    or dense (.csv).

    Example:
        X_train, X_test, y_train, y_test = load_split("cup98", "7030")
    """

    path = os.path.join(base_path, split)

    # ---- Load X_train ----
    npz_path = os.path.join(path, f"X_train_{data}.npz")
    csv_path = os.path.join(path, f"X_train_{data}.csv")

    if os.path.exists(npz_path):
        X_train = sparse.load_npz(npz_path)
    else:
        X_train = pd.read_csv(csv_path)

    # ---- Load X_test ----
    npz_path = os.path.join(path, f"X_test_{data}.npz")
    csv_path = os.path.join(path, f"X_test_{data}.csv")

    if os.path.exists(npz_path):
        X_test = sparse.load_npz(npz_path)
    else:
        X_test = pd.read_csv(csv_path)

    # ---- Load labels ----
    y_train = pd.read_csv(os.path.join(path, f"y_train_{data}.csv"))
    y_test  = pd.read_csv(os.path.join(path, f"y_test_{data}.csv"))

    # Convert DataFrames → Series
    y_train = y_train.iloc[:, 0]
    y_test  = y_test.iloc[:, 0]

    return X_train, X_test, y_train, y_test



def load_all_by_split(datasets, base_path="../data/splits/"):
    split_types = ["7030", "3070", "5050"]
    result = {split: {} for split in split_types}

    for split in split_types:
        print(f"\n=== Loading {split} splits ===")
        for data in datasets:
            print(f"  -> Loading {data}")
            X_train, X_test, y_train, y_test = load_split(data, split, base_path)
            result[split][data] = {
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test
            }

    return result


def evaluate_boosting_model(model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy":  accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall":    recall_score(y_test, preds),
        "f1_score":  f1_score(y_test, preds),
        "auc_roc":   roc_auc_score(y_test, probs),
    }

In [27]:
# boosting_models.py (or just a cell in your notebook)

import os
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


# -------------------------------------------------------------------
# 1. Helpers: data prep + metrics
# -------------------------------------------------------------------

def _prepare_xy(split_dict):
    """
    Takes one entry like all_splits["3070"]["wine"] and returns
    dense float32 X_train/X_test and int32 y_train/y_test.
    """
    X_train = split_dict["X_train"]
    X_test  = split_dict["X_test"]
    y_train = split_dict["y_train"]
    y_test  = split_dict["y_test"]

    # Sparse → dense if needed
    if hasattr(X_train, "toarray"):
        X_train = X_train.toarray()
        X_test  = X_test.toarray()

    X_train = np.asarray(X_train, dtype=np.float32)
    X_test  = np.asarray(X_test,  dtype=np.float32)

    # y as int32
    if hasattr(y_train, "to_numpy"):
        y_train = y_train.to_numpy()
        y_test  = y_test.to_numpy()
    y_train = np.asarray(y_train, dtype=np.int32)
    y_test  = np.asarray(y_test,  dtype=np.int32)

    return X_train, X_test, y_train, y_test


def _evaluate_probs(y_true, y_pred, y_prob):
    """Return metrics dict from labels + probs."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "auc_roc": auc,
    }


# -------------------------------------------------------------------
# 2. BAGDT*  (Bagged Decision Trees - NOT boosting)
# -------------------------------------------------------------------

def _fit_bagged_trees(X_train, y_train, X_test, y_test):
    """
    BAGDT*: 100 ID3-style trees with node stop at < 50 samples.
    Implemented with XGBoost as an approximation:
      - n_estimators = 100
      - learning_rate = 1.0 (no boosting shrinkage)
      - max_depth = 20
      - min_child_weight = 50
    """

    model = XGBClassifier(
        n_estimators=100,
        max_depth=20,
        min_child_weight=50,
        learning_rate=1.0,      # no boosting shrinkage → bagging-like
        subsample=1.0,
        colsample_bytree=1.0,
        tree_method="hist",
        device="cuda",
        objective="binary:logistic",
        eval_metric="logloss",
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = _evaluate_probs(y_test, y_pred, y_prob)

    return {
        "model_family": "BAGDT",
        "best_iters": 100,           # number of trees
        "n_estimators": 100,
        "max_depth": 20,
        "min_child_weight": 50,
        "learning_rate": 1.0,
        "model": model,
        **metrics,
    }


# -------------------------------------------------------------------
# 3. Boosting families: BSTDT* (trees) and BSTST* (stumps)
# -------------------------------------------------------------------

def _fit_boosting_family(
    X_train,
    y_train,
    X_test,
    y_test,
    iteration_list,
    max_depth,
    model_family,
    min_child_weight,
    learning_rate=0.1,
):
    """
    Generic boosting wrapper:
      - Uses validation split to pick best iters from iteration_list.
      - Retrains final model on full X_train, evaluates on X_test.
    """

    print(f"\n### {model_family}: selecting best boosting rounds")

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train,
    )

    best_acc = -1.0
    best_iters = None

    # Validation sweep over {2^i}
    for iters in iteration_list:
        model = XGBClassifier(
            n_estimators=iters,
            max_depth=max_depth,
            min_child_weight=min_child_weight,
            learning_rate=learning_rate,
            subsample=1.0,
            colsample_bytree=1.0,
            tree_method="hist",
            device="cuda",
            objective="binary:logistic",
            eval_metric="logloss",
        )

        model.fit(X_tr, y_tr)
        preds_val = model.predict(X_val)
        acc_val = accuracy_score(y_val, preds_val)

        print(f"  {model_family}: iters={iters:5d} → val acc = {acc_val:.4f}")

        if acc_val > best_acc:
            best_acc = acc_val
            best_iters = iters

    print(f"Selected {model_family} iterations: {best_iters}")

    # Retrain on full training set with best_iters
    final_model = XGBClassifier(
        n_estimators=best_iters,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        learning_rate=learning_rate,
        subsample=1.0,
        colsample_bytree=1.0,
        tree_method="hist",
        device="cuda",
        objective="binary:logistic",
        eval_metric="logloss",
    )

    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    y_prob = final_model.predict_proba(X_test)[:, 1]

    metrics = _evaluate_probs(y_test, y_pred, y_prob)

    print(f"\n### {model_family}: final test metrics")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    return {
        "model_family": model_family,
        "best_iters": best_iters,
        "n_estimators": best_iters,
        "max_depth": max_depth,
        "min_child_weight": min_child_weight,
        "learning_rate": learning_rate,
        "model": final_model,
        **metrics,
    }


# -------------------------------------------------------------------
# 4. CSV logging helper
# -------------------------------------------------------------------

def _append_rows_to_csv(rows, csv_path):
    """
    rows: list of flat dicts (same keys)
    """
    df = pd.DataFrame(rows)

    file_exists = os.path.exists(csv_path)
    df.to_csv(csv_path, mode="a", header=not file_exists, index=False)


# -------------------------------------------------------------------
# 5. Main entry: boost_pipeline(all_splits, split, dataset)
# -------------------------------------------------------------------

def boost_pipeline(all_splits, split, dataset, results_csv="boosting_results.csv", save_csv=True):
    """
    all_splits: dict like all_splits["3070"]["wine"] → split dict
    split:     "3070", "5050", "7030"
    dataset:   "wine", "customer", "cup98"

    Returns:
        out: {
          "dataset": str,
          "split": str,
          "bagdt": {...},
          "bstdt": {...},
          "bstst": {...},
          "rows": [flat row dicts for CSV]
        }
    """

    print(f"\n\n===== {dataset.upper()} {split} =====")

    split_dict = all_splits[split][dataset]
    X_train, X_test, y_train, y_test = _prepare_xy(split_dict)

    rows = []

    # ---------------- BAGDT* ----------------
    bag = _fit_bagged_trees(X_train, y_train, X_test, y_test)
    rows.append({
        "dataset": dataset,
        "split": split,
        "model": bag["model_family"],
        "best_iters": bag["best_iters"],
        "n_estimators": bag["n_estimators"],
        "max_depth": bag["max_depth"],
        "min_child_weight": bag["min_child_weight"],
        "learning_rate": bag["learning_rate"],
        "accuracy": bag["accuracy"],
        "precision": bag["precision"],
        "recall": bag["recall"],
        "f1_score": bag["f1_score"],
        "auc_roc": bag["auc_roc"],
    })

    # ---------------- BSTDT* (boosted trees) ----------------
    BSTDT_ITERS = [2**i for i in range(11)]  # 1 .. 1024
    bstdt = _fit_boosting_family(
        X_train, y_train, X_test, y_test,
        iteration_list=BSTDT_ITERS,
        max_depth=20,
        model_family="BSTDT",
        min_child_weight=50,
        learning_rate=0.1,
    )
    rows.append({
        "dataset": dataset,
        "split": split,
        "model": bstdt["model_family"],
        "best_iters": bstdt["best_iters"],
        "n_estimators": bstdt["n_estimators"],
        "max_depth": bstdt["max_depth"],
        "min_child_weight": bstdt["min_child_weight"],
        "learning_rate": bstdt["learning_rate"],
        "accuracy": bstdt["accuracy"],
        "precision": bstdt["precision"],
        "recall": bstdt["recall"],
        "f1_score": bstdt["f1_score"],
        "auc_roc": bstdt["auc_roc"],
    })

    # ---------------- BSTST* (boosted stumps) ----------------
    BSTST_ITERS = [2**i for i in range(16)]  # 1 .. 32768
    bstst = _fit_boosting_family(
        X_train, y_train, X_test, y_test,
        iteration_list=BSTST_ITERS,
        max_depth=1,                # stump
        model_family="BSTST",
        min_child_weight=1,
        learning_rate=0.1,
    )
    rows.append({
        "dataset": dataset,
        "split": split,
        "model": bstst["model_family"],
        "best_iters": bstst["best_iters"],
        "n_estimators": bstst["n_estimators"],
        "max_depth": bstst["max_depth"],
        "min_child_weight": bstst["min_child_weight"],
        "learning_rate": bstst["learning_rate"],
        "accuracy": bstst["accuracy"],
        "precision": bstst["precision"],
        "recall": bstst["recall"],
        "f1_score": bstst["f1_score"],
        "auc_roc": bstst["auc_roc"],
    })

    # Save all three rows to CSV (append)
    if save_csv:
        _append_rows_to_csv(rows, results_csv)
        print(f"\nAppended results → {results_csv}")

    return {
        "dataset": dataset,
        "split": split,
        "bagdt": bag,
        "bstdt": bstdt,
        "bstst": bstst,
        "rows": rows,
    }


# -------------------------------------------------------------------
# 6. Optional helper to run everything in one shot
# -------------------------------------------------------------------

def run_all_boosting(all_splits, results_csv="boosting_results.csv"):
    """
    Convenience: runs boost_pipeline for all
    splits ∈ {3070, 5050, 7030} and datasets ∈ {wine, customer, cup98}.
    """
    all_rows = []
    for split in ["3070", "5050", "7030"]:
        for dataset in ["wine", "customer", "cup98"]:
            out = boost_pipeline(all_splits, split=split, dataset=dataset, results_csv=results_csv, save_csv=True)
            all_rows.extend(out["rows"])
    return pd.DataFrame(all_rows)


In [28]:
datasets = ["wine", "cup98", "customer"]

all_splits = load_all_by_split(datasets)



=== Loading 7030 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 3070 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 5050 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer


In [29]:
df_boost = run_all_boosting(all_splits)
df_boost.head()




===== WINE 3070 =====

### BSTDT: selecting best boosting rounds
  BSTDT: iters=    1 → val acc = 0.7538
  BSTDT: iters=    2 → val acc = 0.7538
  BSTDT: iters=    4 → val acc = 0.7538
  BSTDT: iters=    8 → val acc = 0.9282
  BSTDT: iters=   16 → val acc = 0.9385
  BSTDT: iters=   32 → val acc = 0.9769
  BSTDT: iters=   64 → val acc = 0.9692
  BSTDT: iters=  128 → val acc = 0.9692
  BSTDT: iters=  256 → val acc = 0.9692
  BSTDT: iters=  512 → val acc = 0.9692
  BSTDT: iters= 1024 → val acc = 0.9692
Selected BSTDT iterations: 32

### BSTDT: final test metrics
accuracy: 0.9760
precision: 0.9847
recall: 0.9170
f1_score: 0.9496
auc_roc: 0.9942

### BSTST: selecting best boosting rounds
  BSTST: iters=    1 → val acc = 0.7538
  BSTST: iters=    2 → val acc = 0.7538
  BSTST: iters=    4 → val acc = 0.7538
  BSTST: iters=    8 → val acc = 0.9282
  BSTST: iters=   16 → val acc = 0.9385
  BSTST: iters=   32 → val acc = 0.9718
  BSTST: iters=   64 → val acc = 0.9846
  BSTST: iters=  128 → val

/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



### BSTDT: final test metrics
accuracy: 0.9492
precision: 0.0000
recall: 0.0000
f1_score: 0.0000
auc_roc: 0.5544

### BSTST: selecting best boosting rounds
  BSTST: iters=    1 → val acc = 0.9493
  BSTST: iters=    2 → val acc = 0.9493
  BSTST: iters=    4 → val acc = 0.9493
  BSTST: iters=    8 → val acc = 0.9493
  BSTST: iters=   16 → val acc = 0.9493
  BSTST: iters=   32 → val acc = 0.9493
  BSTST: iters=   64 → val acc = 0.9493
  BSTST: iters=  128 → val acc = 0.9493
  BSTST: iters=  256 → val acc = 0.9493
  BSTST: iters=  512 → val acc = 0.9493
  BSTST: iters= 1024 → val acc = 0.9493
  BSTST: iters= 2048 → val acc = 0.9492
  BSTST: iters= 4096 → val acc = 0.9488
  BSTST: iters= 8192 → val acc = 0.9485
  BSTST: iters=16384 → val acc = 0.9467
  BSTST: iters=32768 → val acc = 0.9448
Selected BSTST iterations: 1


/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



### BSTST: final test metrics
accuracy: 0.9492
precision: 0.0000
recall: 0.0000
f1_score: 0.0000
auc_roc: 0.5665

Appended results → boosting_results.csv


===== WINE 5050 =====

### BSTDT: selecting best boosting rounds
  BSTDT: iters=    1 → val acc = 0.7538
  BSTDT: iters=    2 → val acc = 0.7538
  BSTDT: iters=    4 → val acc = 0.8477
  BSTDT: iters=    8 → val acc = 0.9369
  BSTDT: iters=   16 → val acc = 0.9615
  BSTDT: iters=   32 → val acc = 0.9846
  BSTDT: iters=   64 → val acc = 0.9785
  BSTDT: iters=  128 → val acc = 0.9800
  BSTDT: iters=  256 → val acc = 0.9800
  BSTDT: iters=  512 → val acc = 0.9800
  BSTDT: iters= 1024 → val acc = 0.9800
Selected BSTDT iterations: 32

### BSTDT: final test metrics
accuracy: 0.9809
precision: 0.9817
recall: 0.9400
f1_score: 0.9604
auc_roc: 0.9947

### BSTST: selecting best boosting rounds
  BSTST: iters=    1 → val acc = 0.7538
  BSTST: iters=    2 → val acc = 0.7538
  BSTST: iters=    4 → val acc = 0.7538
  BSTST: iters=    8 → val acc 

...


### BSTST: final test metrics
accuracy: 0.9492
precision: 0.0000
recall: 0.0000
f1_score: 0.0000
auc_roc: 0.5636

Appended results → boosting_results.csv


===== WINE 7030 =====

### BSTDT: selecting best boosting rounds
  BSTDT: iters=    1 → val acc = 0.7538
  BSTDT: iters=    2 → val acc = 0.7538
  BSTDT: iters=    4 → val acc = 0.9011
  BSTDT: iters=    8 → val acc = 0.9516
  BSTDT: iters=   16 → val acc = 0.9648
  BSTDT: iters=   32 → val acc = 0.9747
  BSTDT: iters=   64 → val acc = 0.9802
  BSTDT: iters=  128 → val acc = 0.9780
  BSTDT: iters=  256 → val acc = 0.9780
  BSTDT: iters=  512 → val acc = 0.9780
  BSTDT: iters= 1024 → val acc = 0.9780
Selected BSTDT iterations: 64

### BSTDT: final test metrics
accuracy: 0.9877
precision: 0.9872
recall: 0.9625
f1_score: 0.9747
auc_roc: 0.9964

### BSTST: selecting best boosting rounds
  BSTST: iters=    1 → val acc = 0.7538
  BSTST: iters=    2 → val acc = 0.7538
  BSTST: iters=    4 → val acc = 0.7538
  BSTST: iters=    8 → val acc 

/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



### BSTDT: final test metrics
accuracy: 0.9492
precision: 0.0000
recall: 0.0000
f1_score: 0.0000
auc_roc: 0.5823

### BSTST: selecting best boosting rounds
  BSTST: iters=    1 → val acc = 0.9492
  BSTST: iters=    2 → val acc = 0.9492
  BSTST: iters=    4 → val acc = 0.9492
  BSTST: iters=    8 → val acc = 0.9492
  BSTST: iters=   16 → val acc = 0.9492
  BSTST: iters=   32 → val acc = 0.9492
  BSTST: iters=   64 → val acc = 0.9492
  BSTST: iters=  128 → val acc = 0.9492
  BSTST: iters=  256 → val acc = 0.9492
  BSTST: iters=  512 → val acc = 0.9492
  BSTST: iters= 1024 → val acc = 0.9492
  BSTST: iters= 2048 → val acc = 0.9493
  BSTST: iters= 4096 → val acc = 0.9493
  BSTST: iters= 8192 → val acc = 0.9493
  BSTST: iters=16384 → val acc = 0.9492
  BSTST: iters=32768 → val acc = 0.9489
Selected BSTST iterations: 2048

### BSTST: final test metrics
accuracy: 0.9492
precision: 0.0000
recall: 0.0000
f1_score: 0.0000
auc_roc: 0.6178

Appended results → boosting_results.csv


/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,dataset,split,model,best_iters,n_estimators,max_depth,min_child_weight,learning_rate,accuracy,precision,recall,f1_score,auc_roc
0,wine,3070,BAGDT,100,100,20,50,1.0,0.956034,0.923573,0.895536,0.909338,0.980744
1,wine,3070,BSTDT,32,32,20,50,0.1,0.976039,0.984660,0.916964,0.949607,0.994247
2,wine,3070,BSTST,256,256,1,1,0.1,0.993185,0.987466,0.984821,0.986142,0.997216
3,customer,3070,BAGDT,100,100,20,50,1.0,0.676596,0.728863,0.649351,0.686813,0.725743
4,customer,3070,BSTDT,8,8,20,50,0.1,0.680142,0.718194,0.681818,0.699534,0.719071
